# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yousseftarektalkhan2006-blip/The-ML-internsip/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Review pages first when they are old and still receive substantial search visibility

Staleness — MIXED: Pages 91–180 days since update have a higher declining rate than pages updated within 90 days, but the 181–365 day bucket reverses the pattern. The 365+ bucket contains only 5 pages, so it is too small to support a conclusion.

Search visibility — CONFIRMED: Pages with medium, high, and very high search visibility have substantially higher declining rates than pages with low visibility. This supports using search visibility as a signal for prioritizing pages for review.

In [15]:
!rm -rf The-ML-internsip
!git clone https://github.com/yousseftarektalkhan2006-blip/The-ML-internsip.git

Cloning into 'The-ML-internsip'...
remote: Enumerating objects: 138, done.
remote: Counting objects: 100% (138/138), done.
remote: Compressing objects: 100% (95/95), done.
remote: Total 138 (delta 50), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (138/138), 1.86 MiB | 8.46 MiB/s, done.
Resolving deltas: 100% (50/50), done.


In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.



import pandas as pd

df = pd.read_csv("The-ML-internsip/data/raw/content_refresh_anonymized.csv")

print(df.shape)
print(df.columns.tolist())

print(
    df[
        [
            "days_since_last_update",
            "impressions_90d"
        ]
    ].head()
)

(30000, 44)
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
   days_since_last_update  impressions_90d
0                      20             3803
1                      25            15320
2                      20            12581
3                      22            11751
4 

In [17]:
df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

print(df["is_declining_label"].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [21]:
df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 90, 180, 365, float("inf")],
    labels=["0-90 days", "91-180 days", "181-365 days", "365+ days"]
)

staleness_table = (
    df.groupby("staleness_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

print(staleness_table)

  staleness_bucket      n  declining_rate
0        0-90 days  20655        0.512031
1      91-180 days   9171        0.611057
2     181-365 days    169        0.467456
3        365+ days      5        0.600000


In [22]:
df["visibility_bucket"] = pd.qcut(
    df["impressions_90d"],
    q=4,
    labels=["Low", "Medium", "High", "Very high"],
    duplicates="drop"
)

visibility_table = (
    df.groupby("visibility_bucket", observed=False)
      .agg(
          n=("is_declining_label", "size"),
          declining_rate=("is_declining_label", "mean")
      )
      .reset_index()
)

print(visibility_table)

  visibility_bucket     n  declining_rate
0               Low  7503        0.376116
1            Medium  7499        0.604614
2              High  7498        0.625634
3         Very high  7500        0.562000


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
stale = (df["days_since_last_update"] >= 91).astype(int)

visible = df["visibility_bucket"].isin(
    ["High", "Very high"]
).astype(int)

df["score"] = stale * visible

df["reason_code"] = ""

df.loc[df["score"] == 1, "reason_code"] = "stale_and_visible"

df["action"] = ""

df.loc[df["score"] == 1, "action"] = "review_for_refresh"

ranked_queue = df.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

import os

os.makedirs(
    "The-ML-internsip/work/outputs",
    exist_ok=True
)

ranked_queue.to_csv(
    "The-ML-internsip/work/outputs/baseline_action_score.csv",
    index=False
)

print(ranked_queue[[
    "content_id",
    "score",
    "reason_code",
    "action"
]].head(10))

print("\nScore counts:")
print(ranked_queue["score"].value_counts())

print("\nCSV written successfully.")

             content_id  score        reason_code              action
0  content_c50ae59fc471      1  stale_and_visible  review_for_refresh
1  content_eea80eacb125      1  stale_and_visible  review_for_refresh
2  content_44098c7327f3      1  stale_and_visible  review_for_refresh
3  content_86d53cec4dd6      1  stale_and_visible  review_for_refresh
4  content_2aa11e7419fa      1  stale_and_visible  review_for_refresh
5  content_9c2b02889e9c      1  stale_and_visible  review_for_refresh
6  content_5378ff3b5f5b      1  stale_and_visible  review_for_refresh
7  content_6aa43079fb0c      1  stale_and_visible  review_for_refresh
8  content_3f2c294ef17d      1  stale_and_visible  review_for_refresh
9  content_fc2a4854f0fe      1  stale_and_visible  review_for_refresh

Score counts:
score
0    24012
1     5988
Name: count, dtype: int64

CSV written successfully.


In [30]:
# Rank the pages that passed our original rule
priority_pages = df[df["score"] == 1].copy()


# Rank by staleness:
# older pages should receive higher priority
priority_pages["staleness_rank"] = priority_pages[
    "days_since_last_update"
].rank(
    method="min",
    ascending=True
)

priority_pages["staleness_rank"] = (
    len(priority_pages) + 1
    - priority_pages["staleness_rank"]
)


# Rank by visibility:
# pages with more impressions should receive higher priority
priority_pages["visibility_rank"] = priority_pages[
    "impressions_90d"
].rank(
    method="min",
    ascending=False
)

priority_pages["visibility_rank"] = (
    len(priority_pages) + 1
    - priority_pages["visibility_rank"]
)


# Give equal importance to staleness and visibility
priority_pages["priority_score"] = (
    priority_pages["staleness_rank"]
    + priority_pages["visibility_rank"]
) / 2


# Create the final ranked queue
ranked_queue = priority_pages.sort_values(
    by="priority_score",
    ascending=False
).reset_index(drop=True)


# Make sure the required output directory exists
import os

os.makedirs(
    "The-ML-internsip/work/outputs",
    exist_ok=True
)


# Write the ranked queue
ranked_queue.to_csv(
    "The-ML-internsip/work/outputs/baseline_action_score.csv",
    index=False
)


# Show the first 10 pages
print(
    ranked_queue[
        [
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "staleness_rank",
            "visibility_rank",
            "priority_score",
            "reason_code",
            "action"
        ]
    ].head(10)
)

print("\nNumber of pages in the priority queue:", len(ranked_queue))
print("\nCSV written successfully.")

             content_id  days_since_last_update  impressions_90d  \
0  content_5fe46e04994d                     104           517715   
1  content_2dba2b1f9536                     104           443434   
2  content_2c2606c5d176                     104           347399   
3  content_cb112fce36be                     104           309910   
4  content_9532f197bbc8                     104           309192   
5  content_36ff89c8214e                     104           295097   
6  content_b28d1efd668f                     104           286608   
7  content_813e88069237                     104           233561   
8  content_c21024970297                     104           211366   
9  content_c8e9d6ab9013                     104           208678   

   staleness_rank  visibility_rank  priority_score        reason_code  \
0          5961.0           5988.0          5974.5  stale_and_visible   
1          5961.0           5987.0          5974.0  stale_and_visible   
2          5961.0           5986

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## 3. Top-20 review

1. Review for refresh — stale and highly visible. Could be wrong if freshness is not the main issue.
2. Review for refresh — stale and highly visible. Could be wrong because its position is relatively weak.
3. Review for refresh — stale and highly visible. Could be wrong if the page already performs well.
4. Review for refresh — stale and highly visible. Could be wrong if the issue is transactional intent.
5. Review for refresh — stale and highly visible. Could be wrong because its position and CTR are already strong.
6. Review for refresh — stale and highly visible. Could be wrong if low CTR is caused by ranking rather than freshness.
7. Review for refresh — stale and highly visible. Could be wrong because its position is weak.
8. Review for refresh — stale and highly visible. Could be wrong if the commercial intent needs another solution.
9. Review for refresh — stale and highly visible. Could be wrong if performance is already adequate.
10. Review for refresh — stale and highly visible. Weak pick because clicks and engagement are all zero.
11. Review for refresh — stale and highly visible. Could be wrong because its position is weak.
12. Review for refresh — stale and highly visible. Could be wrong if engagement is the main issue.
13. Review for refresh — stale and highly visible. Could be wrong if CTR is the main issue.
14. Review for refresh — stale and highly visible. Could be wrong if position is the main issue.
15. Review for refresh — stale and highly visible. Could be wrong if the page already performs well.
16. Review for refresh — stale and highly visible. Could be wrong if the page already performs well.
17. Review for refresh — stale and highly visible. Could be wrong if engagement is the main issue.
18. Review for refresh — stale and highly visible. Could be wrong because recent visibility is increasing.
19. Review for refresh — stale and highly visible. Could be wrong because its position is weak.
20. Review for refresh — stale and highly visible. Could be wrong if freshness is not the main issue.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

top20 = ranked_queue.head(20)

print(
    top20[
        [
            "content_id",
            "days_since_last_update",
            "impressions_90d",
            "staleness_rank",
            "visibility_rank",
            "priority_score",
            "reason_code",
            "action",
            "content_type",
            "main_intent",
            "content_age_days",
            "days_since_last_update",
            "impressions_last_30d",
            "impressions_prev_30d",
            "clicks_90d",
            "ctr",
            "avg_position",
            "engagement_rate",
            "scroll_rate"
        ]
    ].to_string(index=False)
)

          content_id  days_since_last_update  impressions_90d  staleness_rank  visibility_rank  priority_score       reason_code             action    content_type   main_intent  content_age_days  days_since_last_update  impressions_last_30d  impressions_prev_30d  clicks_90d  ctr  avg_position  engagement_rate  scroll_rate
content_5fe46e04994d                     104           517715          5961.0           5988.0          5974.5 stale_and_visible review_for_refresh keyword article informational               537                     104                120791                218786         741 0.14           4.2             4.23        26.90
content_2dba2b1f9536                     104           443434          5961.0           5987.0          5974.0 stale_and_visible review_for_refresh keyword article informational               299                     104                139891                137909         910 0.21          27.9             2.73         7.77
content_2c2606c5d176     

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak pick

`content_c8e9d6ab9013` is a weak pick because it has high visibility and is stale, but it has 0 clicks, 0% CTR, and 0 engagement/scroll rates.

### Leakage check

The baseline score uses only `days_since_last_update` and `impressions_90d`. It does not use the decline label, `trend_direction`, `trend_pct`, or future-window data.

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.